# Prepare and merge all data for Autoencoder

In [1]:
import pandas as pd
import re
from datetime import datetime

In [2]:
# Variables and Paths
ALL_DATA_CSV = "output/merged_data.csv"
LATENT_FILE = "output/Experiments/BetaScanVAE/latent_representations/train_mu_beta_9.00e-05.csv"
# LATENT_FILE = "output/Experiments/BetaScanVAE/latent_representations/train_mu_beta_3.00e-06.csv"
# LATENT_FILE = "output/Experiments/BetaScanVAE/latent_representations/train_mu_beta_3.00e-05.csv"
# LATENT_FILE = "output/Experiments/BetaScanVAE/latent_representations/val_mu_beta_3.00e-05.csv"
DICOM_FILE = "data/csvData/dicom_metadata.csv"
OUTPUT_FILE = "output/final_train_combined_vae_data_beta_9.00e-05.csv"
# OUTPUT_FILE = "output/final_validation_combined_vae_data.csv"

In [3]:
# Load merged clinical data
df_merged = pd.read_csv(ALL_DATA_CSV, low_memory=False)
print(f"Merged clinical data: {df_merged.shape}")

# Load latent vectors
df_latent = pd.read_csv(LATENT_FILE)
print(f"Latent vectors: {df_latent.shape}")

# Load DICOM metadata for scanner info
df_dicom = pd.read_csv(DICOM_FILE)
print(f"DICOM metadata: {df_dicom.shape}")

Merged clinical data: (41816, 42)
Latent vectors: (2380, 259)
DICOM metadata: (2986, 6)


In [4]:
df_latent.rename(columns={'file_path': 'FilePath'}, inplace=True)
df_latent_clean = df_latent.dropna(subset=['FilePath']).copy()
df_latent_clean.shape

(2380, 259)

In [5]:
df_latent_clean.head(2)

,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,latent_246,latent_247,latent_248,latent_249,latent_250,latent_251,latent_252,latent_253,latent_254,latent_255
0,data/Images/PPMI_Images_PD/219605/Reconstructe...,219605,PD,0.013135,0.021049,0.022535,-0.015517,0.034482,-0.015343,0.076105,...,0.008032,-0.040575,0.109967,-0.031761,0.662258,-0.082733,0.048495,-0.000827,-0.002468,0.056098
1,data/Images/PPMI_Images_PD/3502/Reconstructed_...,3502,PD,0.061054,-0.018390,-0.000745,-0.013741,0.025024,-0.015248,0.098051,...,-0.012984,-0.115652,-0.160568,0.014093,0.432014,-0.132076,0.032982,0.015312,-0.016438,0.004294


In [6]:
df_dicom.rename(columns={'file_path': 'FilePath'}, inplace=True)
df_dicom_latest = df_dicom.dropna(subset=['FilePath']).copy()
df_dicom_latest.shape

(2986, 6)

In [7]:
df_dicom_latest.head(2)

,FilePath,group,PatientSex,StudyDescription,Manufacturer,ManufacturerModelName
0,Images\PPMI_Images_PD\100001\Reconstructed_DaT...,PD,M,1-DAT,SIEMENS NM,Encore2
1,Images\PPMI_Images_PD\100001\Reconstructed_DaT...,PD,M,V02-DAT,SIEMENS NM,Encore2


In [8]:
print("--- Latent DataFrame Path Example ---")
print(df_latent_clean['FilePath'].iloc[0])

print("\n--- DICOM DataFrame Path Example ---")
print(df_dicom_latest['FilePath'].iloc[0])

--- Latent DataFrame Path Example ---
data/Images/PPMI_Images_PD/219605/Reconstructed_DaTSCAN/2023-04-06_14_52_54.0/I1698019/PPMI_219605_NM_Reconstructed_DaTSCAN_Br_20230508143813985_1_S1220401_I1698019.dcm

--- DICOM DataFrame Path Example ---
Images\PPMI_Images_PD\100001\Reconstructed_DaTSCAN\2020-09-09_17_07_33.0\I1452480\PPMI_100001_NM_Reconstructed_DaTSCAN_Br_20210608102518754_1_S1028880_I1452480.dcm


In [9]:
# Function to normalize paths
def normalize_path(path_str):
    if pd.isna(path_str): return path_str
    
    # 1. Force forward slashes
    clean_p = path_str.replace('\\', '/')
    
    # 2. Remove 'data/' prefix if it exists to ensure matching
    if clean_p.startswith('data/'):
        clean_p = clean_p.replace('data/', '')
        
    # 3. Strip any leading/trailing whitespace
    return clean_p.strip()

# Apply to BOTH dataframes
df_latent_clean['Merge_Key'] = df_latent_clean['FilePath'].apply(normalize_path)
df_dicom_latest['Merge_Key'] = df_dicom_latest['FilePath'].apply(normalize_path)

# Check if they look the same now
print("New Key Latent:", df_latent_clean['Merge_Key'].iloc[0])
print("New Key DICOM: ", df_dicom_latest['Merge_Key'].iloc[0])

New Key Latent: Images/PPMI_Images_PD/219605/Reconstructed_DaTSCAN/2023-04-06_14_52_54.0/I1698019/PPMI_219605_NM_Reconstructed_DaTSCAN_Br_20230508143813985_1_S1220401_I1698019.dcm
New Key DICOM:  Images/PPMI_Images_PD/100001/Reconstructed_DaTSCAN/2020-09-09_17_07_33.0/I1452480/PPMI_100001_NM_Reconstructed_DaTSCAN_Br_20210608102518754_1_S1028880_I1452480.dcm


In [10]:
df_latent_with_scanner = pd.merge(
    df_latent_clean,
    df_dicom_latest[['Merge_Key', 'Manufacturer', 'ManufacturerModelName']],
    on='Merge_Key',
    how='left'  # Keep all latent vectors, add scanner info
)
df_latent_with_scanner.shape

(2380, 262)

In [11]:
df_latent_with_scanner['FilePath'] = df_latent_with_scanner['Merge_Key']

In [12]:
df_latent_with_scanner.sample(5)

,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,latent_249,latent_250,latent_251,latent_252,latent_253,latent_254,latent_255,Merge_Key,Manufacturer,ManufacturerModelName
2137,Images/PPMI_Images_Cont/106126/Reconstructed_D...,106126,Control,0.007952,0.019709,0.020150,0.052669,-0.020566,0.048626,0.000242,...,0.029700,-0.933287,0.026338,-0.001903,-0.043766,-0.040448,0.051242,Images/PPMI_Images_Cont/106126/Reconstructed_D...,SIEMENS NM,Encore2
461,Images/PPMI_Images_PD/4030/Reconstructed_DaTSC...,4030,PD,0.031740,-0.022880,0.006123,-0.013079,0.010467,0.053407,0.087451,...,0.010767,-0.204126,-0.114954,-0.024707,0.019282,0.019070,0.066194,Images/PPMI_Images_PD/4030/Reconstructed_DaTSC...,SIEMENS NM,Encore2
1472,Images/PPMI_Images_PD/40771/Reconstructed_DaTS...,40771,PD,-0.138038,0.012953,-0.020731,-0.018318,0.015078,-0.024873,-0.060963,...,-0.073780,-1.107654,0.178326,-0.065104,-0.059513,-0.027222,0.063064,Images/PPMI_Images_PD/40771/Reconstructed_DaTS...,GE MEDICAL SYSTEMS,MILLENNIUM MPS
731,Images/PPMI_Images_PD/3233/Reconstructed_DaTSC...,3233,PD,0.003090,0.013328,-0.003701,-0.001483,0.027420,0.025354,-0.002811,...,0.006989,-0.201943,0.014456,-0.008651,-0.033540,0.024022,0.049510,Images/PPMI_Images_PD/3233/Reconstructed_DaTSC...,SIEMENS NM,Encore2
2111,Images/PPMI_Images_PD/3448/Reconstructed_DaTSC...,3448,PD,0.031946,-0.027885,0.018159,0.035583,-0.009104,0.114537,0.304457,...,0.001795,0.070929,0.026749,0.042447,-0.007489,-0.022137,0.058068,Images/PPMI_Images_PD/3448/Reconstructed_DaTSC...,"Marconi Medical Systems, NM Division",P3000XP


In [13]:
# 1. Robust Date Extraction (Finds YYYY-MM-DD anywhere in path)
def get_date_from_path(path_str):
    if pd.isna(path_str):
        return None
    
    # Regex to find pattern: 4 digits - 2 digits - 2 digits
    match = re.search(r'(\d{4}-\d{2}-\d{2})', str(path_str))
    if match:
        date_raw = match.group(1) # Extracts '2021-04-06'
        try:
            # Added datetime import requirement and better error handling
            return datetime.strptime(date_raw, '%Y-%m-%d').strftime('%m/%Y')
        except Exception:
            return None
    return None

In [14]:
# 2. Apply the fix
df_latent_with_scanner['DATSCAN_DATE'] = df_latent_with_scanner['FilePath'].apply(get_date_from_path)

# Verify we actually have dates now (Safe check)
dates_found = df_latent_with_scanner['DATSCAN_DATE'].dropna()
if not dates_found.empty:
    print("Latent Date Sample:", dates_found.iloc[0])
else:
    print("Warning: No dates could be extracted from FilePath. Check your regex or path format.")

df_latent_with_scanner.sample(2)

Latent Date Sample: 04/2023


,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,latent_250,latent_251,latent_252,latent_253,latent_254,latent_255,Merge_Key,Manufacturer,ManufacturerModelName,DATSCAN_DATE
2234,Images/PPMI_Images_PD/3482/Reconstructed_DaTSC...,3482,PD,0.013318,0.031805,0.055944,-0.004605,0.035620,-0.006984,0.039675,...,1.141409,0.012326,0.023782,-0.038833,0.013943,0.026621,Images/PPMI_Images_PD/3482/Reconstructed_DaTSC...,SIEMENS NM,IP2,03/2014
840,Images/PPMI_Images_PD/70818/Reconstructed_DaTS...,70818,PD,0.002617,0.013192,0.026815,0.001454,-0.044694,0.031115,-0.005336,...,-0.167331,0.085235,0.014843,-0.071422,0.002899,0.000050,Images/PPMI_Images_PD/70818/Reconstructed_DaTS...,GE MEDICAL SYSTEMS,INFINIA,02/2022


In [15]:

# 3. Clean Clinical Data (df_merged)
df_merged['PATNO'] = pd.to_numeric(df_merged['PATNO'], errors='coerce').fillna(0).astype(int)
df_latent_with_scanner['PATNO'] = pd.to_numeric(df_latent_with_scanner['PATNO'], errors='coerce').fillna(0).astype(int)

In [16]:
df_latent_with_scanner.sample(2)

,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,latent_250,latent_251,latent_252,latent_253,latent_254,latent_255,Merge_Key,Manufacturer,ManufacturerModelName,DATSCAN_DATE
911,Images/PPMI_Images_PD/142356/Reconstructed_DaT...,142356,PD,-0.029270,0.018728,0.016379,-0.028174,0.003921,0.019020,-0.004487,...,0.232495,-0.066738,0.028817,0.004114,0.013708,0.038348,Images/PPMI_Images_PD/142356/Reconstructed_DaT...,GE MEDICAL SYSTEMS,INFINIA,03/2022
2363,Images/PPMI_Images_PD/4052/Reconstructed_DaTSC...,4052,PD,-0.015918,0.009664,0.008237,0.085671,0.041531,0.115359,-0.125467,...,0.583981,0.148685,0.112406,-0.048875,-0.012173,0.008403,Images/PPMI_Images_PD/4052/Reconstructed_DaTSC...,"Marconi Medical Systems, NM Division",3000XP,05/2012


In [17]:
df_latent_with_scanner.shape

(2380, 263)

In [18]:
# Convert clinical dates to strings, handle NaNs
df_merged['DATSCAN_DATE'] = pd.to_datetime(
    df_merged['DATSCAN_DATE'], 
    format='mixed', 
    errors='coerce'
).dt.strftime('%m/%Y')

df_merged.sample(2)

,PATNO,EVENT_ID,AGE_AT_VISIT,BIRTHDT,SEX,INFODT,REC_ID,PAG_NAME,AFICBERB,ASHKJEW,...,DATSCAN_DATE,DATSCAN_CAUDATE_R,DATSCAN_CAUDATE_L,DATSCAN_PUTAMEN_R,DATSCAN_PUTAMEN_L,DATSCAN_PUTAMEN_R_ANT,DATSCAN_PUTAMEN_L_ANT,DATSCAN_ANALYZED,DATSCAN_NOT_ANALYZED_REASON,DATSCAN_OTHER_SPECIFY
26028,114613,V04,58.4,07/1964,1.0,10/2021,IA88826,SCREEN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40595,356931,BL,73.1,04/1952,1.0,04/2025,IA714667,SCREEN,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
df_merged.shape

(41816, 42)

In [20]:
# 4. Perform the Merge
df_combined = pd.merge(
    df_latent_with_scanner, 
    df_merged,
    on=['PATNO', 'DATSCAN_DATE'],
    how='inner'
)

print(f"Merge Shape: {df_combined.shape}")

Merge Shape: (2373, 303)


In [21]:
df_combined['DATSCAN_DATE'].sample(5)

1596    01/2013
2350    08/2012
1643    10/2012
515     12/2011
2216    07/2014
Name: DATSCAN_DATE, dtype: object

In [22]:
df_combined.sample(5)

,FilePath,PATNO,label,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,...,DATSCAN_LIGAND,DATSCAN_CAUDATE_R,DATSCAN_CAUDATE_L,DATSCAN_PUTAMEN_R,DATSCAN_PUTAMEN_L,DATSCAN_PUTAMEN_R_ANT,DATSCAN_PUTAMEN_L_ANT,DATSCAN_ANALYZED,DATSCAN_NOT_ANALYZED_REASON,DATSCAN_OTHER_SPECIFY
1380,Images/PPMI_Images_Cont/3213/Reconstructed_DaT...,3213,Control,0.051893,0.052997,-0.020285,-0.016534,-0.017972,0.045672,0.060802,...,NaN,2.67,2.47,1.54,1.39,2.05,1.99,Yes,NaN,NaN
1493,Images/PPMI_Images_PD/100898/Reconstructed_DaT...,100898,PD,-0.064303,0.004588,0.032323,0.028110,0.018169,0.099293,0.027752,...,123I-DaTscan,0.91,1.03,0.54,0.64,0.71,0.85,Yes,NaN,NaN
529,Images/PPMI_Images_PD/3462/Reconstructed_DaTSC...,3462,PD,0.011838,0.000647,-0.046653,0.024459,0.067303,0.028103,-0.001099,...,NaN,1.56,2.36,0.65,0.89,1.07,1.47,Yes,NaN,NaN
1795,Images/PPMI_Images_PD/143119/Reconstructed_DaT...,143119,PD,-0.104092,0.055568,0.125794,-0.060653,0.050940,-0.015243,0.138795,...,123I-DaTscan,1.74,1.78,0.72,1.32,1.19,1.37,Yes,NaN,NaN
267,Images/PPMI_Images_Cont/3544/Reconstructed_DaT...,3544,Control,-0.022547,0.031668,0.041417,-0.072413,0.021305,0.047717,0.185807,...,NaN,2.22,2.93,1.45,1.91,2.07,2.38,Yes,NaN,NaN


In [23]:
"Manufacturer" in df_combined.columns

True

In [24]:
# Save the final merged dataset for the Autoencoder

df_combined.to_csv(OUTPUT_FILE, index=False)

print(f"Successfully saved merged data to: {OUTPUT_FILE}")
print(f"Final file contains {df_combined.shape[0]} rows and {df_combined.shape[1]} columns.")

Successfully saved merged data to: output/final_train_combined_vae_data_beta_9.00e-05.csv
Final file contains 2373 rows and 303 columns.
